# Recommendation System

This notebook develops the content-based homestay recommendation system by:

1. Loading the prepared dataset.
2. Converting homestay feature representations into TF-IDF vectors.
3. Computing similarity scores using cosine similarity.
4. Generating personalized homestay recommendations.
5. Providing explainable recommendations based on shared features.
6. Testing the recommendation engine on sample homestays.

The generated recommendation model will be used for evaluation and deployment.

In [24]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

In [25]:
# ==========================================
# LOAD PREPARED DATASET
# ==========================================

df = pd.read_csv(
    "../data/final/homestays_prepared.csv"
)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (1157, 39)


,homestay_id,homestay_name,owner_name,category,district,block,village,owner_email,owner_mobile,google_name,...,breakfast,mountain_view,room_service,bonfire_barbeque,pickup_dropoff_service,description,price_band,amenity_text,location_text,feature_text
0,1,Revere Homestay,Mr. Riwaj Pradhan,Silver,Kalimpong,Municipality,"8Th Mile, Kalimpong",riwajpradhan10@gmail.com,9800686780,Revere Homsetay,...,1,0,0,0,0,"Revere Homestay in 8th Mile, Kalimpong Village...",mid_range,parking breakfast,"Kalimpong Municipality 8Th Mile, Kalimpong","Revere Homestay in 8th Mile, Kalimpong Village..."
1,2,Mansarover Homestay,Miss Tina Mani Gurung,Gold,Kalimpong,Municipality,"Chandralok, Kalimpong",santabgurung53@gmail.com,9932234895,Mansarover Homestay / Flora & Transport,...,0,0,0,1,0,"Mansarover Homestay in Chandralok Village, Mun...",premium,wifi parking bonfire_barbeque,"Kalimpong Municipality Chandralok, Kalimpong","Mansarover Homestay in Chandralok Village, Mun..."
2,3,Bethany Homestay,Anupama Tamang,Silver,Kalimpong,Kalimpong I,Dr.Grahams Home Block B,wangchuck20199@gmial.com,8348993048,Bethany Homestay Kalimpong,...,1,1,1,1,0,BETHANY HOMESTAY is positioned within Kalimpon...,budget,parking breakfast mountain_view room_service b...,Kalimpong Kalimpong I Dr.Grahams Home Block B,BETHANY HOMESTAY is positioned within Kalimpon...
3,4,S3 Homestay,Sangita Rai,Silver,Kalimpong,Kalimpong I,Upper Echhey Dara Gaon Kalimpong,sangitasankalp@gmail.com,9933410313,Sunrise Inn Homestay,...,1,1,1,1,0,"S3 Homestay in Upper Echhey Dara Gaon, Kalimpo...",mid_range,parking breakfast mountain_view room_service b...,Kalimpong Kalimpong I Upper Echhey Dara Gaon K...,"S3 Homestay in Upper Echhey Dara Gaon, Kalimpo..."
4,5,Bajarangi Homestay,Kamal Kumar Sharma,Silver,Kalimpong,Kalimpong I,Singi Samalbong Kalimpong,bajrangihomestay@gmail.com,8670450557,Bajrangi Homestay,...,1,1,0,1,0,BAJARANGI HOMESTAY in SINGI SAMALBONG KALIMPON...,mid_range,wifi parking breakfast mountain_view bonfire_b...,Kalimpong Kalimpong I Singi Samalbong Kalimpong,BAJARANGI HOMESTAY in SINGI SAMALBONG KALIMPON...


In [26]:
# ==========================================
# FEATURE TEXT VALIDATION
# ==========================================

print(
    df.loc[0, "feature_text"]
)

Revere Homestay in 8th Mile, Kalimpong Village offers convenient access to town, Deolo, and Durpin. Enjoy essential amenities including free Wi-Fi, parking, breakfast, mountain views, and pickup/drop-off service. Rated 5.0 with one review, this Silver-category homestay suits travelers seeking proximity to key locations and reliable, well-located accommodations. parking breakfast Kalimpong Municipality 8Th Mile, Kalimpong Silver mid_range


In [27]:
# ==========================================
# TF-IDF VECTORIZATION
# ==========================================

tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    df["feature_text"]
)

print(
    "TF-IDF Matrix Shape:",
    tfidf_matrix.shape
)

TF-IDF Matrix Shape: (1157, 2088)


In [28]:
# ==========================================
# COSINE SIMILARITY MATRIX
# ==========================================

cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

print(
    "Similarity Matrix Shape:",
    cosine_sim.shape
)

Similarity Matrix Shape: (1157, 1157)


In [29]:
# ==========================================
# HOMESTAY INDEX MAPPING
# ==========================================

indices = pd.Series(
    df.index,
    index=df["homestay_name"]
).drop_duplicates()

print(
    "Total Homestays:",
    len(indices)
)

Total Homestays: 1157


In [30]:
# ==========================================
# RECOMMENDATION FUNCTION
# ==========================================

def recommend_homestays(
    homestay_name,
    top_n=5
):

    idx = indices[homestay_name]

    similarity_scores = list(
        enumerate(
            cosine_sim[idx]
        )
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[
        1:top_n + 1
    ]

    homestay_indices = [
        i[0]
        for i in similarity_scores
    ]

    scores = [
        round(i[1], 3)
        for i in similarity_scores
    ]

    recommendations = (
        df.iloc[homestay_indices]
        .copy()
    )

    recommendations[
        "similarity_score"
    ] = scores

    return recommendations

In [31]:
# ==========================================
# TF-IDF EXPLAINABILITY FUNCTIONS
# ==========================================

feature_names = tfidf.get_feature_names_out()

def get_top_matching_terms(
    source_idx,
    recommended_idx,
    top_n=5
):
    
    source_vector = (
        tfidf_matrix[source_idx]
        .toarray()[0]
    )

    recommended_vector = (
        tfidf_matrix[recommended_idx]
        .toarray()[0]
    )

    contributions = (
        source_vector
        *
        recommended_vector
    )

    top_indices = (
        contributions.argsort()
        [-top_n:]
        [::-1]
    )

    terms = [
        feature_names[i]
        for i in top_indices
        if contributions[i] > 0
    ]

    return terms


def generate_explanation(
    source_row,
    recommended_row,
    source_idx,
    recommended_idx
):

    reasons = []

    # ==========================
    # Common Amenities
    # ==========================

    amenity_columns = [

        "wifi",
        "parking",
        "breakfast",
        "mountain_view",
        "room_service",
        "bonfire_barbeque",
        "pickup_dropoff_service"

    ]

    common_features = []

    for amenity in amenity_columns:

        if (
            source_row[amenity] == 1
            and
            recommended_row[amenity] == 1
        ):

            common_features.append(
                amenity.replace("_", " ")
            )

    # ==========================
    # Location Similarity
    # ==========================

    if (
        source_row["block"]
        ==
        recommended_row["block"]
    ):
        reasons.append(
            f"Both are located in {source_row['block']}"
        )

    # ==========================
    # Category Similarity
    # ==========================

    if (
        source_row["category"]
        ==
        recommended_row["category"]
    ):
        reasons.append(
            f"Same category ({source_row['category']})"
        )

    # ==========================
    # Rating Similarity
    # ==========================

    if abs(
        source_row["rating"]
        -
        recommended_row["rating"]
    ) <= 0.5:

        reasons.append(
            "Similar ratings"
        )

    # ==========================
    # Price Similarity
    # ==========================

    if abs(
        source_row["price"]
        -
        recommended_row["price"]
    ) <= 1000:

        reasons.append(
            "Similar price range"
        )

    # ==========================
    # TF-IDF Terms
    # ==========================

    tfidf_terms = get_top_matching_terms(
        source_idx,
        recommended_idx
    )

    return (
        common_features,
        reasons,
        tfidf_terms
    )

In [32]:
# ==========================================
# EXPLAINABLE RECOMMENDATIONS
# ==========================================

def explainable_recommendations(
    homestay_name,
    top_n=5
):

    recommendations = recommend_homestays(
        homestay_name,
        top_n
    )

    source_idx = indices[
        homestay_name
    ]

    source_row = df.iloc[
        source_idx
    ]

    print(
        f"\nSelected Homestay: {homestay_name}"
    )

    print("=" * 80)

    for _, row in recommendations.iterrows():

        recommended_idx = row.name

        print(
            f"\nRecommended: {row['homestay_name']}"
        )

        print(
            f"Similarity Score: "
            f"{row['similarity_score']:.3f}"
        )

        print(
            f"Rating: {row['rating']}"
        )

        print(
            f"Price: ₹{row['price']}"
        )

        (
            common_features,
            additional_reasons,
            tfidf_terms
        ) = generate_explanation(
            source_row,
            row,
            source_idx,
            recommended_idx
        )

        print("\nCommon Features:")

        if common_features:

            for feature in common_features:

                print(
                    f"✓ {feature.title()}"
                )

        print("\nModel Explanation:")

        if tfidf_terms:

            print(
                "Top matching TF-IDF terms:"
            )

            for term in tfidf_terms:

                print(
                    f"• {term}"
                )

        print("\nAdditional Similarities:")

        for reason in additional_reasons:

            print(
                f"✓ {reason}"
            )

        # =====================
        # Natural Language XAI
        # =====================

        feature_text = ", ".join(
            common_features[:3]
        )

        term_text = ", ".join(
            tfidf_terms[:5]
        )

        explanation = (

            f"{row['homestay_name']} was "
            f"recommended because it shares "
            f"similar characteristics with "
            f"{homestay_name}, including "
            f"{feature_text}. "

            f"The recommendation is further "
            f"supported by similar descriptive "
            f"terms such as {term_text}."

        )

        print(
            "\nNatural Language Explanation:"
        )

        print(explanation)

        print("-" * 80)

In [33]:
# ==========================================
# TEST RECOMMENDATIONS
# ==========================================

sample_homestay = (
    df["homestay_name"]
    .iloc[0]
)

recommend_homestays(
    sample_homestay
)[
    [
        "homestay_name",
        "rating",
        "price",
        "similarity_score"
    ]
]

,homestay_name,rating,price,similarity_score
166,Windsong Homestay,4.5,2501,0.520
155,The Birds View Homestay,4.6,3163,0.364
1127,Kalash Villa Homestay,4.3,1267,0.340
336,Jublee Homestay,4.7,1853,0.295
165,Pranati Residency Homestay,4.9,1112,0.295


In [34]:
# ==========================================
# TEST EXPLAINABILITY
# ==========================================

sample_homestay = (
    df["homestay_name"]
    .iloc[0]
)

explainable_recommendations(
    sample_homestay
)


Selected Homestay: Revere Homestay

Recommended: Windsong Homestay
Similarity Score: 0.520
Rating: 4.5
Price: ₹2501

Common Features:
✓ Parking
✓ Breakfast

Model Explanation:
Top matching TF-IDF terms:
• 8th
• mile
• kalimpong
• mid_range
• parking

Additional Similarities:
✓ Similar ratings
✓ Similar price range

Natural Language Explanation:
Windsong Homestay was recommended because it shares similar characteristics with Revere Homestay, including parking, breakfast. The recommendation is further supported by similar descriptive terms such as 8th, mile, kalimpong, mid_range, parking.
--------------------------------------------------------------------------------

Recommended: The Birds View Homestay
Similarity Score: 0.364
Rating: 4.6
Price: ₹3163

Common Features:
✓ Parking
✓ Breakfast

Model Explanation:
Top matching TF-IDF terms:
• 8th
• mile
• municipality
• locations
• key

Additional Similarities:
✓ Both are located in Municipality
✓ Similar ratings

Natural Language Explana

In [35]:
# ==========================================
# PREFERENCE FILTERING
# ==========================================

def search_by_preferences(

    mountain_view=False,
    breakfast=False,
    parking=False,
    block=None

):

    results = df.copy()

    if mountain_view:

        results = results[
            results["mountain_view"] == 1
        ]

    if breakfast:

        results = results[
            results["breakfast"] == 1
        ]

    if parking:

        results = results[
            results["parking"] == 1
        ]

    if block:

        results = results[
            results["block"]
            ==
            block
        ]

    return results[
        [
            "homestay_name",
            "rating",
            "price",
            "block"
        ]
    ].sort_values(
        by="rating",
        ascending=False
    )

In [36]:
search_by_preferences(
    mountain_view=True,
    breakfast=True,
    parking=True
).head()

,homestay_name,rating,price,block
1152,Valley View Homestay,5.0,1757,Kalimpong I
1101,Sewa Kunj Eco Stay,5.0,1493,Kalimpong I
1093,Kelsang Homestay,5.0,1127,Kalimpong I
1058,Holiday Homestay,5.0,1468,Kalimpong I
1085,The Bethlehem Inn,5.0,2076,Gorubathan


In [37]:
# ==========================================
# SAVE MODEL FILES
# ==========================================

with open(
    "../models/tfidf_vectorizer.pkl",
    "wb"
) as f:

    pickle.dump(
        tfidf,
        f
    )

with open(
    "../models/cosine_similarity.pkl",
    "wb"
) as f:

    pickle.dump(
        cosine_sim,
        f
    )

with open(
    "../models/indices.pkl",
    "wb"
) as f:

    pickle.dump(
        indices,
        f
    )

with open(
    "../models/tfidf_matrix.pkl",
    "wb"
) as f:

    pickle.dump(
        tfidf_matrix,
        f
    )
    
print(
    "Model files saved successfully."
)

Model files saved successfully.
